
# How To Polars 

In [ ]:
# read data
# load cleaned data
import pickle 
with open("transcripts_speech_clean.pickle", "rb") as f:
    transcripts = pickle.load(f)

transcripts

In [ ]:
import polars as pl

# there are two main data structures in polars Series and Dataframes
# Series are one-dimensional and homogeneous - meaning they contain only one data type
# for example a single column from our dataframe
sp = pl.Series(transcripts["speaker"])
print(sp)

# or just random numbers
ns = pl.Series("numbers", [4, 11, 9, 12224])
print(ns)
# the first attribute in the Series function is the name
# note: variable name != series name
# as you can see, with the "speaker" column the name of the column was adapted as series name


In [ ]:
# Dataframes are two-dimensional and heterogeneous
# we can combine the strings and integers from before and more

df = pl.DataFrame( # to create a df we use a dictionary of column names and lists
    {
        "names": sp[:4], # our series is recognized as list - note that we have to shorten it (first 4 values)
        "numbers": ns,
        "floats": [3.14, 19.8, 0.0, 1.7654]

    }
)
# the output shows the shape (4 rows, 3 columns)
# the name and datatype of the columns and the rows
print(df)

In [ ]:
### Exercise #### 
''' create a df called "authors" with 3 columns; The first column should be called "names" and contain the following values:
"Terry Pratchett", "Wolfgang Goethe", "Charlotte Bronte", "Cornelia Funke"; The next column should be called: "read" 
and contain a Boolean that indicates whether you have read one of their works; call the last column "like" and 
put in how much you like them on a scale of 1 to 5 using integers '''



In [ ]:
# another way of looking at a df
authors.glimpse()

In [ ]:
# this you probably know
authors.head(1) #shows first line
authors.tail(2) # shows two last lines

In [ ]:
# import random
# random.seed(42)
authors.sample(2) # two random rows

### Exercise ####
#try this again a few times with the first two lines uncommented
# can you guess what it does?

In [ ]:
authors.describe() # some statistics for your df

In [ ]:
# back to the df we have created before

# with the "select" function you can
# subset columns
s1 = df.select("numbers", "floats")

# and modify them at the same time 
s2 = df.select(
    n_mean = pl.col("numbers").mean(), # pl.col() is a function that let's you access columns
    biggest_f = pl.col("floats").max(),
)

s3 = df.select(
    pl.col("numbers", "names").n_unique() # even several at once
)

print("Selection of only numbers and names")
print(s1)
print("New modified columns")
print(s2)
print("Selection and modification of Cols")
print(s3)

In [ ]:
# Expressions: define dataframe specific functions

namenumb = pl.col("names") + (pl.col("numbers").cast(pl.String)) 
# with the cast-function you can do typecasting - obviously we can't just calculate the sum of a str and an int
namenumb


In [ ]:
s4 = df.select(
    id = namenumb,
    prod = pl.col("numbers") * pl.col("floats")
)
s4

In [ ]:
# can you guess the difference between with_columns and select?
s5 =  df.with_columns(
    id = namenumb,
    prod = pl.col("numbers") * pl.col("floats")
)
s5

In [ ]:
# filter works a lot like select, but for rows instead of columns
# to filter, we need to tell the system which rows we want to keep
# meaning, we need to fit our criteria in a logical expression

s6 = df.filter(
    pl.col("numbers")>=5,
    pl.col("names") == "RL"
)
s6


In [ ]:
### Exercise ### 
''' 
transform the transcripts df so that it only contains data from session 09
and the columns speaker, speech and event
call this new df "session09"

TIP! you can link several steps together like this:
    result = df.function(
        column = first.operation
    ).function(second.operation)
'''



In [ ]:
# Conditionals
# sometimes we want to operate differently on out data depending on certain conditions
authors_gen = authors.with_columns(
    pl.when(
        pl.col("names").is_in(["Terry Pratchett", "Wolfgang Goethe"])
    ).then(pl.lit("male")) # pl.lit() needed if we just need to pass a string
    .otherwise(pl.lit("female")).alias("gender")
)
authors_gen

In [ ]:
### Exercise ###
'''Take the session09 df and to the event column add 
"Discussion" whenever there's a missing value
TIP: 
use .is_null()
watch out! .otherwise() defaults to NULL so you have to feed back the original values of the column
'''


In [ ]:
# Aggregation 
# we can goup our data and do operations per group like counting how many there are

quota = authors_gen.group_by(
    "gender"
    ).agg(pl.len())
quota

In [ ]:
### Exercise ###
'''For each speaker, count how many times they cite the text
--> event column == "Citing Text" '''



In [ ]:
# Working with Strings
# Regex-based (Regular expressions): formal language to match patterns in strings
# often referred to as "patterns"; basis for text search (ctrl+f) in text editors or 
# search engines, but also forms used on the web (verify whether a string is an 
# email address, a phone number etc)#
# polars offers a range of string functions that use regex: https://docs.pola.rs/api/python/stable/reference/expressions/string.html
# and here's the regex format they use: https://docs.rs/regex/latest/regex/#syntax
teststring = pl.DataFrame({"regex history" : ['Regular expressions originated in 1951, when mathematician Stephen Cole Kleene described regular languages using his mathematical notation called regular events.[6][7]' 'These arose in theoretical computer science, in the subfields of automata theory (models of computation) and the description and classification of formal languages, motivated by Kleene\'s attempt to describe early artificial neural networks.' '(Kleene introduced it as an alternative to McCulloch & Pitts\'s "prehensible", but admitted "We would welcome any suggestions as to a more descriptive term."[8])' 'Other early implementations of pattern matching include the SNOBOL language, which did not use regular expressions, but instead its own pattern matching constructs.']})

numbers = teststring.select(
    pl.col("regex history").str.extract_all(r"(\d+)")
)
print(numbers)

In [ ]:
### Exercise ###
# find all the dates in teststring